# Notebook 02 — Model Implementation and Evaluation
## HCLTech Career Shaper · Personalized Movie Recommendation System

Covers Build Guide Stages 7 (data prep), 8 (baseline), 9 (improve), 10 (evaluate).

**Three algorithms implemented:**
- User-Based Collaborative Filtering (memory-based, cosine similarity)
- Item-Based Collaborative Filtering (memory-based, cosine similarity)
- SVD Matrix Factorization (model-based, scikit-surprise)

**Evaluation metrics:** RMSE and Precision@5

**Key data facts (verified in Sub-Task 2 and split audit):**
- 473 cleaned rows → train 378 / test 95 (80/20, random_state=42)
- 309 users, 8 movies — user-item matrix is ~80.9% sparse
- 51 users appear in both train and test (eligible for Precision@K)
- 39 users appear in test only — cold-start fallback applied
- 2 of 51 eligible users have only 4 unseen movies (fewer than K=5)

In [1]:
import sys, os, json
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import GridSearchCV as SurpriseGridSearchCV

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from recommender import (
    user_cf_predict_rating, user_cf_recommend,
    item_cf_predict_rating, item_cf_recommend,
    svd_predict_rating, svd_recommend,
)

MODELS_DIR = os.path.join(os.getcwd(), '..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
print('Imports OK')

Imports OK


---
## A — Load Data and Create the Single Shared Split

In [2]:
df = pd.read_csv(os.path.join(os.getcwd(), '..', 'data', 'processed', 'cleaned_dataset.csv'))
print(f'Cleaned dataset: {df.shape}')

# --- ONE shared split used by ALL three algorithms ---
train, test = train_test_split(df, test_size=0.2, random_state=42)
train = train.reset_index(drop=True)
test  = test.reset_index(drop=True)

ALL_MOVIES   = sorted(df['movie_title'].unique())
GLOBAL_MEAN  = float(train['rating'].mean())   # computed from training data ONLY
TRAIN_USERS  = set(train['user_id'])
TEST_USERS   = set(test['user_id'])
BOTH_USERS   = sorted(TRAIN_USERS & TEST_USERS)  # eligible for Precision@K
COLD_USERS   = TEST_USERS - TRAIN_USERS

print(f'Train: {len(train)} rows  |  Test: {len(test)} rows')
print(f'Global mean (train only): {GLOBAL_MEAN:.4f}')
print(f'Users in train: {len(TRAIN_USERS)}')
print(f'Users in both train+test (Precision@K eligible): {len(BOTH_USERS)}')
print(f'Cold-start users in test: {len(COLD_USERS)}')
print(f'All movies: {ALL_MOVIES}')

Cleaned dataset: (473, 7)
Train: 378 rows  |  Test: 95 rows
Global mean (train only): 3.6230
Users in train: 270
Users in both train+test (Precision@K eligible): 51
Cold-start users in test: 39
All movies: ['3 Idiots', 'Andhadhun', 'Baahubali', 'Dangal', 'Drishyam', 'KGF', 'Pink', 'RRR']


### A1 — Unseen-candidate distribution for eligible users
Discloses users with fewer than 5 unseen movies (affects Precision@5 denominator policy).

In [3]:
unseen_counts = {}
for u in BOTH_USERS:
    seen   = set(train.loc[train['user_id'] == u, 'movie_title'])
    n_unseen = len([m for m in ALL_MOVIES if m not in seen])
    unseen_counts[u] = n_unseen

dist = pd.Series(unseen_counts.values()).value_counts().sort_index()
print('Unseen-movie count distribution among 51 eligible users:')
print(dist.to_string())
n_lt5 = sum(1 for v in unseen_counts.values() if v < 5)
print(f'\nUsers with < 5 unseen candidates: {n_lt5}')
print('Precision@5 uses fixed denominator=5; users with fewer candidates')
print('can score at most n_unseen/5 — this limitation is documented.')

Unseen-movie count distribution among 51 eligible users:
4     2
5     2
6    13
7    34

Users with < 5 unseen candidates: 2
Precision@5 uses fixed denominator=5; users with fewer candidates
can score at most n_unseen/5 — this limitation is documented.


### A2 — Build training user-item matrix

In [4]:
# Built from TRAINING rows ONLY
user_item_matrix = train.pivot_table(
    index='user_id', columns='movie_title', values='rating'
)
filled = user_item_matrix.notna().sum().sum()
total  = user_item_matrix.shape[0] * user_item_matrix.shape[1]
print(f'Training user-item matrix: {user_item_matrix.shape}')
print(f'Filled cells: {filled} / {total}  ({filled/total*100:.1f}% fill rate)')
print(f'Sparsity: {(1-filled/total)*100:.1f}%')

Training user-item matrix: (270, 8)
Filled cells: 378 / 2160  (17.5% fill rate)
Sparsity: 82.5%


---
## B — Baseline: Global Mean Predictor

In [5]:
# Predict GLOBAL_MEAN for every test row
baseline_preds = np.full(len(test), GLOBAL_MEAN)
baseline_rmse  = float(np.sqrt(mean_squared_error(test['rating'], baseline_preds)))
print(f'Baseline (global mean = {GLOBAL_MEAN:.4f})')
print(f'RMSE on test (95 rows): {baseline_rmse:.4f}')

Baseline (global mean = 3.6230)
RMSE on test (95 rows): 0.8684


---
## C — User-Based Collaborative Filtering

**Approach:** Cosine similarity between user vectors (from training matrix only).  
For each test (user, movie) pair, find the K=10 most similar training users who rated that movie and compute a weighted-average prediction.  
**Cold-start fallback:** training global mean for users absent from the training matrix.

In [6]:
# Build cosine similarity matrix from training matrix (fill NaN with 0 for computation)
matrix_filled = user_item_matrix.fillna(0).values
user_sim_matrix = cosine_similarity(matrix_filled)   # shape: (n_train_users, n_train_users)
print(f'User similarity matrix: {user_sim_matrix.shape}')

User similarity matrix: (270, 270)


In [7]:
# Predict on all 95 test rows
user_cf_preds = []
user_cf_cold  = []
for _, row in test.iterrows():
    pred, is_cold = user_cf_predict_rating(
        row['user_id'], row['movie_title'],
        user_item_matrix, user_sim_matrix,
        train, GLOBAL_MEAN
    )
    user_cf_preds.append(pred)
    user_cf_cold.append(is_cold)

user_cf_rmse = float(np.sqrt(mean_squared_error(test['rating'], user_cf_preds)))
n_cold = sum(user_cf_cold)
print(f'User-CF RMSE on test (95 rows): {user_cf_rmse:.4f}')
print(f'  Cold-start test rows (global-mean fallback): {n_cold}')
print(f'  Warm test rows (CF prediction):              {len(test) - n_cold}')

User-CF RMSE on test (95 rows): 0.9732
  Cold-start test rows (global-mean fallback): 41
  Warm test rows (CF prediction):              54


### C1 — Precision@5 for User-CF

Evaluated on 51 eligible users (present in both train and test).  
Denominator is always 5; users with <5 unseen candidates get top-N where N<5.

In [8]:
def precision_at_k(model_recommend_fn, eligible_users, test_df, k=5, **kwargs):
    """
    Compute mean Precision@K over eligible users.
    
    - eligible_users: users in both train and test
    - test_df: held-out ratings used ONLY for hit/miss checking
    - Relevance threshold: rating >= 4.0
    - Denominator: always k (fixed), even if fewer than k candidates exist
    """
    scores = []
    for u in eligible_users:
        recs = model_recommend_fn(user_id=u, n=k, **kwargs)
        if not recs:
            scores.append(0.0)
            continue
        recommended_movies = [m for m, _, _ in recs]
        # Relevant: user rated this movie >= 4.0 in the TEST set
        user_test = test_df[test_df['user_id'] == u]
        relevant  = set(user_test.loc[user_test['rating'] >= 4.0, 'movie_title'])
        hits = sum(1 for m in recommended_movies if m in relevant)
        scores.append(hits / k)   # fixed denominator k
    return float(np.mean(scores)), scores

user_cf_p5, user_cf_scores = precision_at_k(
    user_cf_recommend,
    BOTH_USERS, test, k=5,
    user_item_matrix=user_item_matrix,
    user_sim_matrix=user_sim_matrix,
    train_df=train,
    global_mean=GLOBAL_MEAN,
    all_movies=ALL_MOVIES,
)
print(f'User-CF Precision@5 (51 eligible users): {user_cf_p5:.4f}')
print(f'  Distribution of per-user P@5: min={min(user_cf_scores):.2f}  '
      f'max={max(user_cf_scores):.2f}  mean={np.mean(user_cf_scores):.4f}')

User-CF Precision@5 (51 eligible users): 0.0745
  Distribution of per-user P@5: min=0.00  max=0.20  mean=0.0745


---
## D — Item-Based Collaborative Filtering

**Approach:** Cosine similarity between item (movie) vectors (transposed training matrix).  
**Cold-start fallback:** same global mean as User-CF.

In [9]:
# Transpose: movies become rows; fill NaN with 0 for cosine computation
item_matrix_filled = user_item_matrix.T.fillna(0).values
item_sim_matrix    = cosine_similarity(item_matrix_filled)  # shape: (8, 8)
print(f'Item similarity matrix: {item_sim_matrix.shape}')
print('Item similarity matrix (movie x movie):')
sim_df = pd.DataFrame(item_sim_matrix, index=ALL_MOVIES, columns=ALL_MOVIES)
print(sim_df.round(3).to_string())

Item similarity matrix: (8, 8)
Item similarity matrix (movie x movie):
           3 Idiots  Andhadhun  Baahubali  Dangal  Drishyam    KGF   Pink    RRR
3 Idiots      1.000      0.084      0.050   0.115     0.059  0.203  0.113  0.080
Andhadhun     0.084      1.000      0.058   0.022     0.057  0.093  0.091  0.113
Baahubali     0.050      0.058      1.000   0.150     0.077  0.075  0.074  0.043
Dangal        0.115      0.022      0.150   1.000     0.058  0.081  0.078  0.074
Drishyam      0.059      0.057      0.077   0.058     1.000  0.083  0.119  0.076
KGF           0.203      0.093      0.075   0.081     0.083  1.000  0.157  0.205
Pink          0.113      0.091      0.074   0.078     0.119  0.157  1.000  0.095
RRR           0.080      0.113      0.043   0.074     0.076  0.205  0.095  1.000


In [10]:
item_cf_preds = []
item_cf_cold  = []
for _, row in test.iterrows():
    pred, is_cold = item_cf_predict_rating(
        row['user_id'], row['movie_title'],
        user_item_matrix, item_sim_matrix,
        train, GLOBAL_MEAN
    )
    item_cf_preds.append(pred)
    item_cf_cold.append(is_cold)

item_cf_rmse = float(np.sqrt(mean_squared_error(test['rating'], item_cf_preds)))
n_cold = sum(item_cf_cold)
print(f'Item-CF RMSE on test (95 rows): {item_cf_rmse:.4f}')
print(f'  Cold-start test rows (global-mean fallback): {n_cold}')
print(f'  Warm test rows (CF prediction):              {len(test) - n_cold}')

Item-CF RMSE on test (95 rows): 1.0134
  Cold-start test rows (global-mean fallback): 41
  Warm test rows (CF prediction):              54


In [11]:
item_cf_p5, item_cf_scores = precision_at_k(
    item_cf_recommend,
    BOTH_USERS, test, k=5,
    user_item_matrix=user_item_matrix,
    item_sim_matrix=item_sim_matrix,
    train_df=train,
    global_mean=GLOBAL_MEAN,
    all_movies=ALL_MOVIES,
)
print(f'Item-CF Precision@5 (51 eligible users): {item_cf_p5:.4f}')
print(f'  Distribution: min={min(item_cf_scores):.2f}  '
      f'max={max(item_cf_scores):.2f}  mean={np.mean(item_cf_scores):.4f}')

Item-CF Precision@5 (51 eligible users): 0.1020
  Distribution: min=0.00  max=0.20  mean=0.1020


---
## E — SVD Matrix Factorization

Uses scikit-surprise. The **same 378 training rows** are loaded into Surprise format.  
The **same 95 test rows** are evaluated — Surprise's own train_test_split is NOT used.  
Hyperparameter tuning uses cross-validation on training data only.  
Cold-start (unknown user in Surprise trainset) → explicit global_mean fallback.

In [12]:
# Load ONLY training rows into Surprise
reader     = Reader(rating_scale=(1, 5))
train_data = Dataset.load_from_df(
    train[['user_id', 'movie_title', 'rating']].copy(),
    reader
)
# Build the full trainset from training data
trainset = train_data.build_full_trainset()
print(f'Surprise trainset: {trainset.n_users} users, {trainset.n_items} items, '
      f'{trainset.n_ratings} ratings')

Surprise trainset: 270 users, 8 items, 378 ratings


In [13]:
# Hyperparameter tuning — training data only, 3-fold CV
param_grid = {
    'n_factors': [20, 50, 100],
    'n_epochs':  [20, 50],
    'lr_all':    [0.005],
    'reg_all':   [0.02],
}
gs = SurpriseGridSearchCV(
    SVD, param_grid, measures=['rmse'], cv=3, refit=True, joblib_verbose=0
)
gs.fit(train_data)   # uses ONLY train_data — test set never seen here

best_params = gs.best_params['rmse']
print(f'Best SVD params (CV on trainset): {best_params}')
print(f'Best CV RMSE (training folds):    {gs.best_score["rmse"]:.4f}')

Best SVD params (CV on trainset): {'n_factors': 100, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02}
Best CV RMSE (training folds):    0.8738


In [14]:
# Refit best model on the FULL training set
best_svd = SVD(
    n_factors=best_params['n_factors'],
    n_epochs=best_params['n_epochs'],
    lr_all=best_params['lr_all'],
    reg_all=best_params['reg_all'],
    random_state=42,
)
best_svd.fit(trainset)
print('SVD refitted on full trainset.')

SVD refitted on full trainset.


In [15]:
# Evaluate on the SAME 95 test rows — convert manually to Surprise predictions
# Explicit cold-start fallback: if was_impossible, use GLOBAL_MEAN
svd_preds   = []
svd_cold    = []
for _, row in test.iterrows():
    pred, is_cold = svd_predict_rating(
        row['user_id'], row['movie_title'], best_svd, GLOBAL_MEAN
    )
    svd_preds.append(pred)
    svd_cold.append(is_cold)

svd_rmse = float(np.sqrt(mean_squared_error(test['rating'], svd_preds)))
n_cold   = sum(svd_cold)
print(f'SVD RMSE on test (95 rows): {svd_rmse:.4f}')
print(f'  Cold-start test rows (global-mean fallback): {n_cold}')
print(f'  Warm test rows (SVD prediction):             {len(test) - n_cold}')

SVD RMSE on test (95 rows): 0.8684
  Cold-start test rows (global-mean fallback): 95
  Warm test rows (SVD prediction):             0


In [16]:
svd_p5, svd_scores = precision_at_k(
    svd_recommend,
    BOTH_USERS, test, k=5,
    svd_algo=best_svd,
    train_df=train,
    all_movies=ALL_MOVIES,
    global_mean=GLOBAL_MEAN,
)
print(f'SVD Precision@5 (51 eligible users): {svd_p5:.4f}')
print(f'  Distribution: min={min(svd_scores):.2f}  '
      f'max={max(svd_scores):.2f}  mean={np.mean(svd_scores):.4f}')

SVD Precision@5 (51 eligible users): 0.1020
  Distribution: min=0.00  max=0.20  mean=0.1020


---
## F — Results Table and Model Comparison

In [17]:
results = pd.DataFrame([
    {
        'Model':          'Global Mean Baseline',
        'RMSE':           round(baseline_rmse, 4),
        'Precision@5':    '—',
        'Test rows':      95,
        'Cold-start rows': 41,
        'Notes':          'Predicts training mean for every test row'
    },
    {
        'Model':          'User-Based CF',
        'RMSE':           round(user_cf_rmse, 4),
        'Precision@5':    round(user_cf_p5, 4),
        'Test rows':      95,
        'Cold-start rows': sum(user_cf_cold),
        'Notes':          'Cosine sim; K=10 neighbours; cold-start → global mean'
    },
    {
        'Model':          'Item-Based CF',
        'RMSE':           round(item_cf_rmse, 4),
        'Precision@5':    round(item_cf_p5, 4),
        'Test rows':      95,
        'Cold-start rows': sum(item_cf_cold),
        'Notes':          'Cosine sim on items; cold-start → global mean'
    },
    {
        'Model':          'SVD',
        'RMSE':           round(svd_rmse, 4),
        'Precision@5':    round(svd_p5, 4),
        'Test rows':      95,
        'Cold-start rows': sum(svd_cold),
        'Notes':          f'Best params: {best_params}; cold-start → global mean'
    },
])

print('=' * 80)
print('EXPERIMENT RESULTS TABLE')
print('=' * 80)
print(results[['Model','RMSE','Precision@5','Cold-start rows','Notes']].to_string(index=False))
print()
print('Precision@5 disclosure:')
print('  - Evaluated on 51 users present in both train and test')
print('  - Relevance threshold: rating >= 4.0 in the test set')
print('  - Fixed denominator K=5; 2 users have only 4 unseen movies (max P@5 = 0.8)')
print('  - With only 8 movies total, K=5 recommends 5 of 8 possible items')
print('  - These metrics will appear higher than real-world CF systems')
print()

# Identify best model
rmse_rows   = results[results['RMSE'] != '—'].copy()
numeric_rmse = results['RMSE'].apply(lambda x: float(x) if x != '—' else float('inf'))
best_rmse_idx = numeric_rmse.idxmin()
print(f'Best RMSE:        {results.loc[best_rmse_idx, "Model"]} ({results.loc[best_rmse_idx, "RMSE"]})')

p5_vals = results['Precision@5'].apply(lambda x: float(x) if x != '—' else -1)
best_p5_idx = p5_vals.idxmax()
print(f'Best Precision@5: {results.loc[best_p5_idx, "Model"]} ({results.loc[best_p5_idx, "Precision@5"]})')
print()
print('Commentary: High sparsity (82.5% in training matrix) and the dominance of')
print('single-rating users limit the personalisation quality of memory-based CF.')
print('SVD decomposes the latent structure and handles sparsity more robustly.')
print('All cold-start predictions (global mean) inflate RMSE equally across models.')

EXPERIMENT RESULTS TABLE
               Model   RMSE Precision@5  Cold-start rows                                                                                                       Notes
Global Mean Baseline 0.8684           —               41                                                                   Predicts training mean for every test row
       User-Based CF 0.9732      0.0745               41                                                       Cosine sim; K=10 neighbours; cold-start → global mean
       Item-Based CF 1.0134       0.102               41                                                               Cosine sim on items; cold-start → global mean
                 SVD 0.8684       0.102               95 Best params: {'n_factors': 100, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02}; cold-start → global mean

Precision@5 disclosure:
  - Evaluated on 51 users present in both train and test
  - Relevance threshold: rating >= 4.0 in the test set
  - Fixed den

---
## G — Save Artefacts

In [18]:
# Save user-item matrix (training only)
joblib.dump(user_item_matrix,  os.path.join(MODELS_DIR, 'user_item_matrix.pkl'))
print('Saved: user_item_matrix.pkl')

# Save similarity matrices
joblib.dump(user_sim_matrix,   os.path.join(MODELS_DIR, 'user_sim_matrix.pkl'))
joblib.dump(item_sim_matrix,   os.path.join(MODELS_DIR, 'item_sim_matrix.pkl'))
print('Saved: user_sim_matrix.pkl')
print('Saved: item_sim_matrix.pkl')

# Save best SVD model
joblib.dump(best_svd,          os.path.join(MODELS_DIR, 'svd_model.pkl'))
print('Saved: svd_model.pkl')

# Save training dataframe (for cold-start detection and fallback in app)
joblib.dump(train,             os.path.join(MODELS_DIR, 'train_df.pkl'))
print('Saved: train_df.pkl')

# Save evaluation results as JSON for use by Streamlit app
eval_results = {
    'global_mean': GLOBAL_MEAN,
    'all_movies':  ALL_MOVIES,
    'baseline':  {'rmse': round(baseline_rmse, 4), 'precision_at_5': None},
    'user_cf':   {'rmse': round(user_cf_rmse, 4),  'precision_at_5': round(user_cf_p5, 4)},
    'item_cf':   {'rmse': round(item_cf_rmse, 4),  'precision_at_5': round(item_cf_p5, 4)},
    'svd':       {'rmse': round(svd_rmse, 4),      'precision_at_5': round(svd_p5, 4)},
    'svd_best_params': best_params,
    'precision_at_5_notes': {
        'eligible_users':        len(BOTH_USERS),
        'cold_start_excluded':   len(COLD_USERS),
        'relevance_threshold':   4.0,
        'denominator':           5,
        'users_with_lt5_unseen': 2,
        'total_movies':          8,
    }
}
with open(os.path.join(MODELS_DIR, 'eval_results.json'), 'w') as f:
    json.dump(eval_results, f, indent=2)
print('Saved: eval_results.json')
print()
print('All artefacts saved to models/')

Saved: user_item_matrix.pkl
Saved: user_sim_matrix.pkl
Saved: item_sim_matrix.pkl
Saved: svd_model.pkl
Saved: train_df.pkl
Saved: eval_results.json

All artefacts saved to models/


---
## Summary

| Artefact | File |
|---|---|
| Training user-item matrix | `models/user_item_matrix.pkl` |
| User cosine similarity matrix | `models/user_sim_matrix.pkl` |
| Item cosine similarity matrix | `models/item_sim_matrix.pkl` |
| Best SVD model | `models/svd_model.pkl` |
| Training dataframe | `models/train_df.pkl` |
| Evaluation results | `models/eval_results.json` |

Ready for Sub-Task 4: Streamlit Application.